# IODA Format

The IODA (Interface for Observation Data Access) format is a standardized data format used within the JEDI framework to store, access, and manipulate observational data. In the JEDI format, observations are organized into groups that separate measured values, errors, metadata, and quality control information. IODA uses NetCDF as the underlying storage format so you can view or manupulate the IODA files using the commands you usually use to view any NetCDF file, with some minor updates to handle "Groups" within the NetCDF file

In order to prepare observations to be used in JEDI we run [IODA-converters](https://github.com/jcsda-internal/ioda-converters) to convert the observation file to the IODA format. IODA-converters are written in Python and are typically structured to:

1. Read raw observation data.
2. Exctract variables and metadata needed by the observation operator or filters
3. Apply necessary transformations or unit conversions.
4. Organize the data into IODA groups.
5. Write out IODA file using `pyioda` (Python bindings to the underlying IODA C++ libraries). 


You can find many examples of IODA-converters in the IODA-converters GitHub repository: https://github.com/jcsda-internal/ioda-converters

### TL;DR
To use JEDI, observations must be in IODA format. You can use an [existing converter](https://github.com/jcsda-internal/ioda-converters)
 or write a new one to convert your observations into the IODA format. Outputs from JEDI that are in observation space, such as feedback or ObsDiag files, are also written in IODA format.


## Example of a IODA file

IODA files are nested NetCDF files that are organized into Groups such as `ObsValue`, `ObsError`, `MetaData`, `PreQC`, etc. 

In [5]:
import xarray as xr
obs_file = xr.open_datatree('../jedi_applications/run_hofx/hofx/inputs/obs/tempo_no2_tropo_20230805T150000Z.nc')
list(obs_file.groups)

['/',
 '/MetaData',
 '/ObsError',
 '/ObsValue',
 '/PreQC',
 '/RetrievalAncillaryData']

In [6]:
obs_file

<xarray.DataTree>
Group: /
│   Dimensions:   (Layer: 72, Location: 8999, Vertice: 73)
│   Coordinates:
│     * Layer     (Layer) int32 288B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
│     * Location  (Location) int64 72kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
│     * Vertice   (Vertice) int32 292B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
│   Attributes:
│       _ioda_layout:          ObsGroup
│       _ioda_layout_version:  0
│       Vertice:               73
│       Location:              8998548
│       converter:             tempo_nc2ioda.py
│       nvars:                 1
│       Layer:                 72
│       sensor:                TEMPO
│       platform:              Intelsat 40e
│       date_time_string:      1980-01-06T00:00:00Z
│       history:               Wed Oct 22 16:09:14 2025: ncks -d Location,1,89985...
│       NCO:                   netCDF Operators version 5.2.4 (Homepage = http://...
├── Group: /MetaData
│       Dimensions:                  (Location: 8999)
│       Data variables:
│           albedo                   (Location) float32 36kB ...
│           cloud_fraction           (Location) float32 36kB ...
│           dateTime                 (Location) datetime64[ns] 72kB ...
│           latitude                 (Location) float32 36kB ...
│           longitude                (Location) float32 36kB ...
│           quality_assurance_value  (Location) float32 36kB ...
│           solar_zenith_angle       (Location) float32 36kB ...
│           viewing_zenith_angle     (Location) float32 36kB ...
├── Group: /ObsError
│       Dimensions:                (Location: 8999)
│       Data variables:
│           nitrogendioxideColumn  (Location) float32 36kB ...
├── Group: /ObsValue
│       Dimensions:                (Location: 8999)
│       Data variables:
│           nitrogendioxideColumn  (Location) float32 36kB ...
├── Group: /PreQC
│       Dimensions:                (Location: 8999)
│       Data variables:
│           nitrogendioxideColumn  (Location) float64 72kB ...
└── Group: /RetrievalAncillaryData
        Dimensions:          (Location: 8999, Layer: 72, Vertice: 73)
        Data variables:
            averagingKernel  (Location, Layer) float32 3MB ...
            pressureVertice  (Location, Vertice) float32 3MB ...

Files that are generated as the JEDI application output may have additional Groups such as `hofx` or `EffectiveQC` but the structure remains similar. You can find more information in the HofX tutorial.

## Running IODA-Converters

Each converter in the IODA-converters repository have a dedidacted unit test that can be used to make sure the converter is working properly and can be used as a reference to learn how the converter runs, what are the requred inputs, and what the outputs look like. Let's take a look at the TROPOMI CO ioda converter. 

In [ioda-converters/test/CMakeLists.txt](https://github.com/JCSDA-internal/ioda-converters/blob/develop/test/CMakeLists.txt) find where TROPOMI CO test is added:


```C++
ecbuild_add_test( TARGET  test_${PROJECT_NAME}_tropomi_co_total
                  TYPE    SCRIPT
                  ENVIRONMENT "PYTHONPATH=${IODACONV_PYTHONPATH}"
                  COMMAND bash
                  ARGS    ${CMAKE_BINARY_DIR}/bin/iodaconv_comp.sh
                          netcdf
                          "${Python3_EXECUTABLE} ${CMAKE_BINARY_DIR}/bin/tropomi_no2_co_nc2ioda.py
                          -i testinput/tropomi_co.nc
                          -o testrun/tropomi_co_total.nc
                          -v co
                          -q 0.5
                          -c total"
                          tropomi_co_total.nc ${IODA_CONV_COMP_TOL})
```

Here you can see the convert's name `tropomi_no2_co_nc2ioda.py` and the inputs used when running this converter. There are additional arguments that are used by the unit test (such as `iodaconv_comp.sh` and `${IODA_CONV_COMP_TOL}` that can be ignored for this discussion.

Following the instruction [here](https://mer-a-o.github.io/howtojedi/discover/) load the JEDI modules and setup your work environment.

```bash
source /discover/nobackup/projects/gmao/advda/swell/jedi_modules/spackstack_1.9_intel
export MPIEXEC=/usr/local/intel/oneapi/2021/mpi/2021.10.0/bin/mpiexec
export JEDI_BUILD=/discover/nobackup/projects/jcsda/s2127/maryamao/geos-esm/jedi-work/build-intel-release/bin
```

Add ioda python bindings to `PYTHONPATH` by running:

```bash
PYTHON_VERSION=`python3 -c 'import sys; version=sys.version_info[:2]; print("{0}.{1}".format(*version))'`
export PYTHONPATH=${JEDI_BUILD}/../lib/python${PYTHON_VERSION}:${PYTHONPATH}
```

A copy of the tropomi ioda-convert and an example input file is available under `jedi_obs` directory of this repository. `cd` into this directory and run the convert:

```bash
python tropomi_no2_co_nc2ioda.py -i obs.tropomi_co_input.nc -o obs.tropomi_co_ioda.nc -v co -c total -q 0.5
```

For this converter `-i` is the input observation file, `-o` is the output observation file in IODA format, `-v` is the variable which can be either `no2` or `co`, `-c` is the column which can be `total` ro `tropo` (for NO2), and `-q` is the `qa_value` used to preflag data that goes into file before QC. 